# 2.1. Data Visualization - Geo Data

The purpose of this notebook is to conduct data analysis on the geographical locations of the launch sites: Vandenberg AFB Space Launch Complex 4, Cape Canaveral Space Launch Complex 40, Kennedy Space Center Launch Complex 39A. In particular, the following are determined:
- Visualization of the launch sites on the map of the USA and corresponding success rate and orbit type
- Closest highways, coastlines, and railways

In [1]:
import folium
import pandas as pd

from pathlib import Path
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [2]:
file_path = Path.cwd().parent / 'data/interim/api-launch-data-table-class.csv'
spacex_df = pd.read_csv(file_path)
spacex_df.head()

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs,class
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.0,2020-11-16,True,True,1
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.4,2020-11-05,True,True,1
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.0,2020-10-24,True,True,1
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.0,2020-10-18,True,True,1
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.0,2020-10-06,True,True,1


In [3]:
# Color map for class -- green if class==1 else red
def color_map_class(outcome):
    color = 'green' if outcome==1 else 'red'
    return color

spacex_df['color_marker'] = spacex_df['class'].map(color_map_class)
spacex_df

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs,class,color_marker
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.000,2020-11-16,True,True,1,green
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.400,2020-11-05,True,True,1,green
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.000,2020-10-24,True,True,1,green
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.000,2020-10-18,True,True,1,green
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.000,2020-10-06,True,True,1,green
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2013-010,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0007,-80.577357,28.561941,0,None EXP,9.630,2013-03-01,False,False,0,red
92,2012-054,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0006,-80.577357,28.561941,0,None EXP,8.249,2012-10-08,False,False,0,red
93,2012-027,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0005,-80.577357,28.561941,0,None EXP,7.379,2012-05-22,False,False,0,red
94,2010-066,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0004,-80.577357,28.561941,0,False PCL,8.774,2010-12-08,False,False,0,red


In [4]:
# Record of coordinates of each launch site
sites_df = spacex_df.groupby('launch_site')[['latitude', 'longitude']].first()
sites_df

,latitude,longitude
launch_site,,
Launch Complex 39A,28.608227,-80.604282
Space Launch Complex 40,28.561941,-80.577357
Space Launch Complex 4E,34.632000,-120.611000


## Map Initialization and Launch Sites

The purpose of this section is to initialize the map and provide markers on the launch sites; including the outcome of each mission conducted in each site.

In [5]:
# Initialize map
initial_coord = [38.5,-97.5]
site_map = folium.Map(initial_coord, zoom_start=5)

# Add marker cluster
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# Add circles with pop-up and markers over each launch site
for site, coords in sites_df.iterrows():
    coords = coords.to_list()

    # Circle marker -- centered at each site
    circle = folium.Circle(coords, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('{}'.format(site)))

    # Marker showing the name for each site
    site_marker = folium.map.Marker(
        coords,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 14; color:#d35400;"><b>%s</b></div>' % '{}'.format(site),
            )
        )

    # Marker cluster to show count of missions - expandable to show pins indicating outcome
    for index, record in spacex_df[spacex_df['launch_site']==site][['orbit','color_marker']].iterrows():
        outcome_marker = folium.Marker(
            location = coords,
            icon = folium.Icon(color='white', icon_color=record['color_marker']),
            popup = record['orbit']
        )
        marker_cluster.add_child(outcome_marker)

    site_map.add_child(circle)
    site_map.add_child(site_marker)
    
site_map

**Findings.** More launches are conducted in the east coast -- SLC 40 or LC 39A -- and the missions are strictly either equatorial orbits with low inclination, or low to high earth orbits. On the other hand, launches conducted in the west coast -- SLC 4E -- are sun synchronous orbits or polar orbits, characterized by high inclination angles. A reason for the large frequency in the east coast is because equatorial orbits like GTOs require a lower amount of energy than that in missions around the polar orbit. To achieve the former, the launch is normally conducted near the equator and directed eastward so that the surface of the earth contributes to an optimal degree to the final speed of the launch. Given the direction and inclination angle of polar orbits, such missions do not get to have the 'free ride' that the Earth's rotation provides; thus requiring more energy.

## Closest highways, railways, and coastlines

Here I plotted markers to indicate the distance of each launch site to the nearest map layers -- transportation infrastructures such as highways and railways, or physical locations such as coast lines.

I first converted the launch sites DataFrame into a geopandas GeoDataFrame, containing the site names and their corresponding longitude and latitude stored together in a shapely `Points` object.

I then created a function that plots the line segments given the data for map layers. This automates distance calculations and plotting on the site map, and all that is needed to be done is to find data for each map layers -- The ones considered are highways, railways, and coastlines. 

In [6]:
from shapely.ops import nearest_points
from shapely import LineString
from shapely.geometry import Point
import topojson as tp
import geopandas as gpd

# GeoPandas DataFrame for Launch Sites
sites_gdf = gpd.GeoDataFrame(
    sites_df, 
    geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude),
    crs='EPSG:4326'
)

def nearest_to_site(sites_gdf: gpd.GeoDataFrame, layers_gdf: gpd.GeoDataFrame, site_map: folium.Map, layer_type: None|str = None, color: str = 'grey') -> None:
    """
    Function to plot line segments to mark distances of launch sites to the
    nearest map layers (e.g., highways, railways, coastlines)

    :param sites_gdf: gpd.GeoDataFrame, GeoDataFrame of launch sites with their corresponding geometries
    
    :param layers_gdf: gpd.GeoDataFrame, GeoDataFrame of map layers with their corresponding geometries
    
    :param site_map: folium.Map, Folium map on which the distance markers are plotted.
    
    :param layer_type: None or str; type of layer -- transportation infrastructures like 'highways' or 'railways', 
                       or physical layers like 'coastlines'. Pass the string to be used as the name of the layer
                       type or pass the column name (as str) in the layers_gdf to be used as the name of the layer.
                       If None is passed, function uses 'layer' by default.

    :param color: str; color for the segment (blue, purple, black, etc.)
                       
    :return: None; the function adds the markers directly to `site_map`.
    """
    
    # Reproject
    sites_proj = sites_gdf.to_crs(epsg=5070)
    layers_proj = layers_gdf.to_crs(epsg=5070)
    
    # Calculate distance
    # Geopandas uses shapely under the hood to calculate the distance
    # Calculated distance is the minimum distance from the point (launch site) to the line (highway)
    for site_name, record in sites_proj[['geometry']].iterrows():
        site_geom = record.values[-1]
    
        #Calculate distance
        distance = layers_proj.geometry.distance(site_geom)
        arg_nearest = distance.idxmin()
        nearest_distance = distance.min()/1000
    
        #Define distance segment
        _, nearest_pt = nearest_points(site_geom, layers_proj.loc[arg_nearest, 'geometry'])
        line_segment = LineString([site_geom, nearest_pt])

        #Calculate Midpoint
        midpoint = gpd.GeoSeries(
            data = line_segment.interpolate(0.5, normalized=True),
            crs = 5070
        ).to_crs(epsg=4326).values[-1]
        
        # Plot marker for nearest layer
        popup = folium.GeoJsonPopup(
            fields = ['description', 'distance'],
            aliases = ['', ''],
            localize = True,
            labels = True,
        )

        if not layer_type:
            layer_type = 'layer'
        else:
            if layer_type in layers_gdf.columns:
                layer_type = layers_gdf.loc[arg_nearest, layer_type]

        markup = f"""
            <a>
                <div style="font-size: 0.8em;">
                <div style="width: 10px;
                            height: 10px;
                            border: 1px solid black;
                            border-radius: 5px;
                            background-color: orange;">
                </div>
            </div>
            </a>
        """
        
        folium.GeoJson(
            gpd.GeoDataFrame(data = dict(geometry = [nearest_pt], 
                                         description = ['Nearest {} to {}'.format(layer_type, site_name)],
                                         distance = ['{:.3f} km'.format(nearest_distance)]),
                            crs=5070).to_crs(epsg=4326),
            popup = popup,
            marker = folium.Marker(icon=folium.DivIcon(html = markup)),
        ).add_to(site_map)
        
        # Plot line segment
        folium.GeoJson(
            gpd.GeoDataFrame(data = dict(geometry = [line_segment]),
                             crs = 5070).to_crs(epsg=4326),
            style_function = lambda x: {'color': color, 'dashArray': "5,10"}
        ).add_to(site_map)

        # Add line marker
        folium.map.Marker(
            [midpoint.y, midpoint.x],
            icon = DivIcon(
                icon_size=(100,100),
                icon_anchor=(0,0),
                html = '<div style="font-size: 14; color:{};"><b>{:.3f} km</b></div>'.format(color, nearest_distance)
            )
        ).add_to(site_map)

### Data for Highways

In [7]:
import requests
import json

us_roadmap_topo = requests.get(
    'https://gist.githubusercontent.com/bricedev/96d2113bd29f60780223/raw/957d51ac88a6de442cf73b9efa8615fce9f9577e/usroads.json'
).json()

file_path = Path.cwd().parent / 'data' / 'raw' / 'us_road_map.json'
with open(file_path, 'w') as f:
    json.dump(us_roadmap_topo, f)

In [8]:
# GeoPandas Dataframe for US Highways -- Extract major highways
roadmap_topo = tp.Topology(us_roadmap_topo, object_name="roads")
roads_gdf = roadmap_topo.to_gdf(crs="EPSG:4326")
roads_gdf = roads_gdf[roads_gdf['type']=='Major Highway'].reset_index(drop=True)

In [9]:
nearest_to_site(sites_gdf, roads_gdf, site_map, layer_type='type', color='purple')

In [10]:
site_map

### Data for Railways

In [11]:
railway_geojson = Path.cwd().parent / 'data/raw/NTAD_North_American_Rail_Network_Lines_-5214657740406327753.geojson'
railway_gdf = gpd.read_file(railway_geojson)
railway_gdf = railway_gdf[railway_gdf['NET'] == 'M'][['geometry']]

In [12]:
nearest_to_site(sites_gdf, railway_gdf, site_map, layer_type='Major Railway', color='#800020')

In [13]:
site_map

### Data for Coastline

#### West Coast

In [14]:
# Launch sites at the west coast
sites_gdf_west = sites_gdf.iloc[2:,:]

# Coast line gdf from geodata that is accurate at the west coast
coastline_path = Path.cwd().parent / "data/raw/ne_10m_coastline.zip"
zip_uri = f"zip://{coastline_path.as_posix()}"
west_coastline_gdf = gpd.read_file(zip_uri)

In [15]:
nearest_to_site(sites_gdf_west, west_coastline_gdf, site_map, layer_type='featurecla', color='blue')

In [16]:
site_map

#### East Coast

In [17]:
# Launch sites at the east coast
sites_gdf_east = sites_gdf.iloc[0:2,:]

# From geodata that is accurate on the east coast
florida_gdf = gpd.read_file(Path.cwd().parent / 'data/raw/Florida_Shoreline_1to12000_Scale_Polygon.geojson')
florida_gdf['geometry'] = florida_gdf.geometry.boundary
sites_gdf = sites_gdf.iloc[1:,:]

In [18]:
nearest_to_site(sites_gdf_east, florida_gdf, site_map, layer_type = 'Coastline', color='blue')

In [19]:
site_map